<a href="https://colab.research.google.com/github/sispo3314/6th-BE-Blog/blob/main/PGPRDT/PGPRDT_v3_2_ucihar_robustness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


변경 사항
- Classifier 입력 축소
  - 이전: torch.cat([ht_pool, hf_pool, hp_pool, alpha_pool])
  - 현재: hp_pool만 사용
- temperature/entropy/consistency loss 제거
- 초반 balance loss 추가

In [ ]:
# ============================================================
# Physics-Guided Patch-Level Time/Frequency Dual Transformer
# for UCI-HAR
#
# Improvements reflected:
#   (1) richer physics descriptors
#   (2) stronger frequency branch (spectral filterbank tokens)
#   (3) patch-level routing instead of sample-level routing
# ============================================================

import os
import math
import copy
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, f1_score

# ============================================================
# 0) Reproducibility / Device
# ============================================================
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============================================================
# 1) Config
# ============================================================
ROOT = "/content/drive/MyDrive/datasets/UCI HAR Dataset"

BATCH_SIZE   = 128
EPOCHS       = 50
LR           = 1e-3
WEIGHT_DECAY = 5e-4
WARMUP_EP    = 5

WINDOW_SIZE  = 128
PATCH_SIZE   = 8                  # 128 -> 16 patches
N_PATCHES    = WINDOW_SIZE // PATCH_SIZE
NUM_FREQ_BANDS = 8                # spectral filterbank bands

D_MODEL      = 64
NHEAD        = 4
NUM_LAYERS   = 2
D_FF         = 128
DROPOUT      = 0.1

GATE_HIDDEN        = 64
LABEL_SMOOTHING    = 0.1

ACTIVITY_NAMES = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]

# ============================================================
# 2) UCI-HAR Loader
# ============================================================
SIGNAL_FILES = [
    "body_acc_x_{}.txt",
    "body_acc_y_{}.txt",
    "body_acc_z_{}.txt",
    "body_gyro_x_{}.txt",
    "body_gyro_y_{}.txt",
    "body_gyro_z_{}.txt",
    "total_acc_x_{}.txt",
    "total_acc_y_{}.txt",
    "total_acc_z_{}.txt",
]

def load_ucihar_split(root, split="train"):
    base = os.path.join(root, split, "Inertial Signals")
    xs = []
    for patt in SIGNAL_FILES:
        path = os.path.join(base, patt.format(split))
        sig = np.loadtxt(path, dtype=np.float32)   # (N, 128)
        xs.append(sig[..., None])                  # (N, 128, 1)
    X = np.concatenate(xs, axis=-1)               # (N, 128, 9)

    y_path = os.path.join(root, split, f"y_{split}.txt")
    y = np.loadtxt(y_path, dtype=np.int64) - 1

    subj_path = os.path.join(root, split, f"subject_{split}.txt")
    subjects = np.loadtxt(subj_path, dtype=np.int64)

    return X, y, subjects

print("\nLoading UCI-HAR...")
X_tr, y_tr, subj_tr = load_ucihar_split(ROOT, "train")
X_te, y_te, subj_te = load_ucihar_split(ROOT, "test")

print("Train:", X_tr.shape, y_tr.shape, "subjects:", np.unique(subj_tr))
print("Test :", X_te.shape, y_te.shape, "subjects:", np.unique(subj_te))

# train-stat normalization
mu  = X_tr.mean(axis=(0, 1), keepdims=True)
std = X_tr.std(axis=(0, 1), keepdims=True) + 1e-8
X_tr = (X_tr - mu) / std
X_te = (X_te - mu) / std

NUM_CLASSES  = len(np.unique(y_tr))
NUM_CHANNELS = X_tr.shape[-1]

print("NUM_CLASSES :", NUM_CLASSES)
print("NUM_CHANNELS:", NUM_CHANNELS)
print("N_PATCHES   :", N_PATCHES)

# ============================================================
# 3) Physics Descriptor Utilities
#    patch-level physics: richer descriptor
# ============================================================
def spectral_entropy_1d(x, eps=1e-8):
    fft_mag = np.abs(np.fft.rfft(x))
    psd = fft_mag ** 2
    psd = psd / (psd.sum() + eps)
    ent = -(psd * np.log(psd + eps)).sum()
    return float(ent / np.log(len(psd) + eps))

def dominant_freq_ratio_1d(x, eps=1e-8):
    fft_mag = np.abs(np.fft.rfft(x))
    power = fft_mag ** 2
    return float(power.max() / (power.sum() + eps))

def low_high_ratio_1d(x, eps=1e-8):
    power = np.abs(np.fft.rfft(x)) ** 2
    if len(power) < 4:
        return 1.0
    mid = max(1, len(power)//3)
    low = power[:mid].sum()
    high = power[mid:].sum()
    return float(low / (high + eps))

def safe_corr(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    if a.std() < 1e-8 or b.std() < 1e-8:
        return 0.0
    return float(np.corrcoef(a, b)[0, 1])

def extract_patch_physics_np(sample, patch_size=8):
    """
    sample: (128, 9)
    return: (N_PATCHES, P_DIM)
    physics dims:
      0 acc_energy
      1 gyro_energy
      2 jerk_energy
      3 spectral_entropy
      4 dominant_freq_ratio
      5 low_high_ratio
      6 gravity_body_consistency
      7 acc_gyro_coupling
      8 axis_correlation
    """
    T, C = sample.shape
    N = T // patch_size
    feats = []

    for i in range(N):
        patch = sample[i*patch_size:(i+1)*patch_size]   # (P, 9)

        body_acc  = patch[:, 0:3]
        body_gyro = patch[:, 3:6]
        total_acc = patch[:, 6:9]

        acc_mag = np.linalg.norm(body_acc, axis=-1)
        gyr_mag = np.linalg.norm(body_gyro, axis=-1)
        tot_mag = np.linalg.norm(total_acc, axis=-1)

        jerk = np.diff(acc_mag, prepend=acc_mag[0])

        acc_energy  = float(np.mean(acc_mag ** 2))
        gyr_energy  = float(np.mean(gyr_mag ** 2))
        jerk_energy = float(np.mean(jerk ** 2))

        spec_ent = 0.5 * (
            spectral_entropy_1d(acc_mag) + spectral_entropy_1d(gyr_mag)
        )
        dom_ratio = 0.5 * (
            dominant_freq_ratio_1d(acc_mag) + dominant_freq_ratio_1d(gyr_mag)
        )
        low_high = 0.5 * (
            low_high_ratio_1d(acc_mag) + low_high_ratio_1d(gyr_mag)
        )

        # gravity/body consistency: total_acc - body_acc roughly gravity-like
        gravity_proxy = total_acc - body_acc
        gravity_mag = np.linalg.norm(gravity_proxy, axis=-1)
        gravity_body_consistency = safe_corr(gravity_mag, tot_mag)

        # acc-gyro coupling
        acc_gyro_coupling = safe_corr(acc_mag, gyr_mag)

        # axis correlation
        axis_corrs = [
            safe_corr(body_acc[:, 0], body_acc[:, 1]),
            safe_corr(body_acc[:, 1], body_acc[:, 2]),
            safe_corr(body_acc[:, 0], body_acc[:, 2]),
        ]
        axis_correlation = float(np.mean(axis_corrs))

        feats.append([
            acc_energy,
            gyr_energy,
            jerk_energy,
            spec_ent,
            dom_ratio,
            low_high,
            gravity_body_consistency,
            acc_gyro_coupling,
            axis_correlation,
        ])

    return np.asarray(feats, dtype=np.float32)

def build_patch_physics_table(X, patch_size=8):
    all_feats = [extract_patch_physics_np(x, patch_size) for x in X]
    all_feats = np.stack(all_feats).astype(np.float32)   # (B, N, P_DIM)
    mu = all_feats.reshape(-1, all_feats.shape[-1]).mean(axis=0, keepdims=True)
    std = all_feats.reshape(-1, all_feats.shape[-1]).std(axis=0, keepdims=True) + 1e-8
    all_feats = (all_feats - mu[None, :, :]) / std[None, :, :]
    return all_feats, mu, std

print("\nBuilding patch-level physics descriptors...")
phys_tr, phys_mu, phys_std = build_patch_physics_table(X_tr, PATCH_SIZE)
phys_te = np.stack([extract_patch_physics_np(x, PATCH_SIZE) for x in X_te]).astype(np.float32)
phys_te = (phys_te - phys_mu[None, :, :]) / phys_std[None, :, :]

PHYS_DIM = phys_tr.shape[-1]
print("Patch physics train:", phys_tr.shape)
print("Patch physics test :", phys_te.shape)
print("PHYS_DIM:", PHYS_DIM)

# ============================================================
# 4) Dataset
# ============================================================
class UCIHARDataset(Dataset):
    def __init__(self, X, y, phys):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
        self.phys = torch.FloatTensor(phys)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.phys[idx], self.y[idx]

tr_ds = UCIHARDataset(X_tr, y_tr, phys_tr)
te_ds = UCIHARDataset(X_te, y_te, phys_te)

tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
te_loader = DataLoader(te_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ============================================================
# 5) Time / Frequency Patch Embedding
# ============================================================
class TimePatchEmbed(nn.Module):
    def __init__(self, num_channels, patch_size, d_model):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Linear(patch_size * num_channels, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        # x: (B, T, C)
        B, T, C = x.shape
        N = T // self.patch_size
        x = x[:, :N*self.patch_size, :]
        x = x.reshape(B, N, self.patch_size * C)
        x = self.proj(x)
        return self.norm(x)

class SpectralFilterbankEmbed(nn.Module):
    """
    Stronger freq branch:
      patch -> rFFT magnitude -> learnable spectral filterbank -> token
    """
    def __init__(self, num_channels, patch_size, num_bands, d_model):
        super().__init__()
        self.patch_size = patch_size
        self.num_channels = num_channels
        self.num_bins = patch_size // 2 + 1
        self.num_bands = num_bands

        # learnable spectral filterbank
        self.filterbank = nn.Parameter(
            torch.randn(num_channels, self.num_bins, num_bands) * 0.02
        )
        self.proj = nn.Linear(num_channels * num_bands, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        # x: (B, T, C)
        B, T, C = x.shape
        N = T // self.patch_size
        x = x[:, :N*self.patch_size, :]
        x = x.reshape(B, N, self.patch_size, C)

        xf = torch.fft.rfft(x, dim=2)         # (B, N, F, C)
        xf = torch.abs(xf)
        xf = xf.permute(0, 1, 3, 2)           # (B, N, C, F)

        # (B, N, C, F) x (C, F, BANDS) -> (B, N, C, BANDS)
        bands = torch.einsum('bncf,cfk->bnck', xf, self.filterbank)
        bands = bands.reshape(B, N, C * self.num_bands)

        z = self.proj(bands)
        return self.norm(z)

# ============================================================
# 6) Transformer Blocks
# ============================================================
class TransformerBlock(nn.Module):
    def __init__(self, d_model, nhead, d_ff, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            dropout=dropout,
            batch_first=True
        )
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x1 = self.norm1(x)
        z, _ = self.attn(x1, x1, x1)
        x = x + z
        x = x + self.ffn(self.norm2(x))
        return x

class BranchEncoder(nn.Module):
    def __init__(self, d_model, nhead, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, nhead, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        return self.norm(x)

# ============================================================
# 7) Patch-Level Physics Gate
# ============================================================
class PatchPhysicsGate(nn.Module):
    """
    alpha_{b,n} = time-branch weight for patch n
    freq weight = 1 - alpha
    """
    def __init__(self, in_dim, hidden=64, temp_init=2.0):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 1)
        )
        self.last_alpha_mean = 0.0

    def forward(self, z_phys_patch):
        # z_phys_patch: (B, N, P_DIM)
        logits = self.mlp(z_phys_patch)              # (B, N, 1)
        alpha = torch.sigmoid(logits)                # (B, N, 1)  soft routing, no temperature
        self.last_alpha_mean = alpha.mean().item()
        return alpha

# ============================================================
# 8) Full Model
# ============================================================
class PhysicsGuidedPatchDualTransformer(nn.Module):
    def __init__(
        self,
        num_channels,
        patch_size,
        num_patches,
        d_model,
        nhead,
        num_layers,
        d_ff,
        dropout,
        num_classes,
        phys_dim,
        gate_hidden=64,
        temp_init=2.0,
        num_freq_bands=8,
    ):
        super().__init__()
        self.num_patches = num_patches
        self.d_model = d_model

        self.time_embed = TimePatchEmbed(num_channels, patch_size, d_model)
        self.freq_embed = SpectralFilterbankEmbed(
            num_channels=num_channels,
            patch_size=patch_size,
            num_bands=num_freq_bands,
            d_model=d_model
        )

        self.time_pos = nn.Parameter(torch.zeros(1, num_patches, d_model))
        self.freq_pos = nn.Parameter(torch.zeros(1, num_patches, d_model))
        nn.init.trunc_normal_(self.time_pos, std=0.02)
        nn.init.trunc_normal_(self.freq_pos, std=0.02)

        self.time_encoder = BranchEncoder(d_model, nhead, d_ff, num_layers, dropout)
        self.freq_encoder = BranchEncoder(d_model, nhead, d_ff, num_layers, dropout)

        self.gate = PatchPhysicsGate(
            in_dim=phys_dim,
            hidden=gate_hidden,
            temp_init=temp_init
        )

        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )

    def forward(self, x, z_phys_patch):
        # x: (B,T,C), z_phys_patch: (B,N,P_DIM)
        ht = self.time_embed(x) + self.time_pos      # (B,N,D)
        hf = self.freq_embed(x) + self.freq_pos      # (B,N,D)

        ht = self.time_encoder(ht)
        hf = self.freq_encoder(hf)

        alpha = self.gate(z_phys_patch)              # (B,N,1)
        h_patch = alpha * ht + (1.0 - alpha) * hf    # patch-wise fusion

        hp_pool = h_patch.mean(dim=1)

        logits = self.classifier(hp_pool)
        return logits, alpha.squeeze(-1), ht, hf, h_patch

    def get_alpha_mean(self):
        return self.gate.last_alpha_mean

# ============================================================
# 9) Perturbation + differentiable patch physics
# ============================================================
def perturb_batch(x, noise_std=0.03, scale_std=0.05, bias_std=0.03):
    B, T, C = x.shape
    s = 1.0 + torch.randn(B, 1, C, device=x.device) * scale_std
    b = torch.randn(B, 1, C, device=x.device) * bias_std
    n = torch.randn_like(x) * noise_std
    return s * x + b + n

def safe_corr_torch(a, b, eps=1e-8):
    a = a - a.mean(dim=1, keepdim=True)
    b = b - b.mean(dim=1, keepdim=True)
    num = (a * b).sum(dim=1)
    den = torch.sqrt((a*a).sum(dim=1) + eps) * torch.sqrt((b*b).sum(dim=1) + eps)
    return num / (den + eps)

def patch_physics_torch(x, patch_size=8):
    """
    x: (B,T,9)
    return: (B,N,P_DIM)
    """
    B, T, C = x.shape
    N = T // patch_size
    x = x[:, :N*patch_size, :].reshape(B, N, patch_size, C)

    body_acc  = x[:, :, :, 0:3]
    body_gyro = x[:, :, :, 3:6]
    total_acc = x[:, :, :, 6:9]

    acc_mag = torch.norm(body_acc, dim=-1)      # (B,N,P)
    gyr_mag = torch.norm(body_gyro, dim=-1)
    tot_mag = torch.norm(total_acc, dim=-1)

    jerk = torch.diff(acc_mag, dim=2, prepend=acc_mag[:, :, :1])

    acc_energy = (acc_mag ** 2).mean(dim=2)
    gyr_energy = (gyr_mag ** 2).mean(dim=2)
    jerk_energy = (jerk ** 2).mean(dim=2)

    # spectral features
    acc_fft = torch.abs(torch.fft.rfft(acc_mag, dim=2)) ** 2
    gyr_fft = torch.abs(torch.fft.rfft(gyr_mag, dim=2)) ** 2

    acc_psd = acc_fft / (acc_fft.sum(dim=2, keepdim=True) + 1e-8)
    gyr_psd = gyr_fft / (gyr_fft.sum(dim=2, keepdim=True) + 1e-8)

    acc_ent = -(acc_psd * torch.log(acc_psd + 1e-8)).sum(dim=2) / math.log(acc_psd.shape[2] + 1e-8)
    gyr_ent = -(gyr_psd * torch.log(gyr_psd + 1e-8)).sum(dim=2) / math.log(gyr_psd.shape[2] + 1e-8)
    spec_ent = 0.5 * (acc_ent + gyr_ent)

    acc_dom = acc_fft.max(dim=2).values / (acc_fft.sum(dim=2) + 1e-8)
    gyr_dom = gyr_fft.max(dim=2).values / (gyr_fft.sum(dim=2) + 1e-8)
    dom_ratio = 0.5 * (acc_dom + gyr_dom)

    mid = max(1, acc_fft.shape[2] // 3)
    acc_lh = acc_fft[:, :, :mid].sum(dim=2) / (acc_fft[:, :, mid:].sum(dim=2) + 1e-8)
    gyr_lh = gyr_fft[:, :, :mid].sum(dim=2) / (gyr_fft[:, :, mid:].sum(dim=2) + 1e-8)
    low_high = 0.5 * (acc_lh + gyr_lh)

    gravity_proxy = total_acc - body_acc
    gravity_mag = torch.norm(gravity_proxy, dim=-1)

    gravity_body_consistency = safe_corr_torch(gravity_mag.reshape(B*N, patch_size), tot_mag.reshape(B*N, patch_size)).reshape(B, N)
    acc_gyro_coupling = safe_corr_torch(acc_mag.reshape(B*N, patch_size), gyr_mag.reshape(B*N, patch_size)).reshape(B, N)

    ax0 = safe_corr_torch(body_acc[:, :, :, 0].reshape(B*N, patch_size), body_acc[:, :, :, 1].reshape(B*N, patch_size)).reshape(B, N)
    ax1 = safe_corr_torch(body_acc[:, :, :, 1].reshape(B*N, patch_size), body_acc[:, :, :, 2].reshape(B*N, patch_size)).reshape(B, N)
    ax2 = safe_corr_torch(body_acc[:, :, :, 0].reshape(B*N, patch_size), body_acc[:, :, :, 2].reshape(B*N, patch_size)).reshape(B, N)
    axis_corr = (ax0 + ax1 + ax2) / 3.0

    z = torch.stack([
        acc_energy,
        gyr_energy,
        jerk_energy,
        spec_ent,
        dom_ratio,
        low_high,
        gravity_body_consistency,
        acc_gyro_coupling,
        axis_corr
    ], dim=-1)   # (B,N,9)
    return z

# ============================================================
# 10) Train / Eval
# ============================================================
def train_model(model, tr_loader, te_loader, epochs, lr, warmup_ep, label):
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)

    def lr_lambda(ep):
        if ep < warmup_ep:
            return (ep + 1) / warmup_ep
        p = (ep - warmup_ep) / max(1, epochs - warmup_ep)
        return 0.5 * (1.0 + np.cos(np.pi * p))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    best_acc = 0.0
    best_state = None
    t0 = time.time()

    for ep in range(1, epochs + 1):
        model.train()
        tr_loss_sum = 0.0
        tr_corr = 0
        tr_n = 0

        for X_b, phys_b, y_b in tr_loader:
            X_b = X_b.to(device)
            phys_b = phys_b.to(device)
            y_b = y_b.to(device)

            optimizer.zero_grad()

            logits, alpha, _, _, _ = model(X_b, phys_b)
            ce_loss = criterion(logits, y_b)

            # weak balance loss in early epochs to prevent gate collapse
            if ep <= 10:
                alpha_mean = alpha.mean()
                balance_loss = (alpha_mean - 0.5).abs()
                loss = ce_loss + 0.01 * balance_loss
            else:
                loss = ce_loss

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            tr_loss_sum += ce_loss.item() * len(y_b)
            tr_corr += (logits.argmax(dim=1) == y_b).sum().item()
            tr_n += len(y_b)

        scheduler.step()

        model.eval()
        te_corr = 0
        te_n = 0
        with torch.no_grad():
            for X_b, phys_b, y_b in te_loader:
                X_b = X_b.to(device)
                phys_b = phys_b.to(device)
                y_b = y_b.to(device)

                logits, _, _, _, _ = model(X_b, phys_b)
                te_corr += (logits.argmax(dim=1) == y_b).sum().item()
                te_n += len(y_b)

        te_acc = te_corr / te_n
        if te_acc > best_acc:
            best_acc = te_acc
            best_state = copy.deepcopy(model.state_dict())

        if ep == 1 or ep % 10 == 0:
            print(
                f"[{label}] ep{ep:3d} | "
                f"loss={tr_loss_sum/max(tr_n,1):.4f} "
                f"tr={tr_corr/max(tr_n,1):.4f} "
                f"te={te_acc:.4f} best={best_acc:.4f} "
                f"alpha_mean={model.get_alpha_mean():.3f}"
            )

    print(f"[{label}] Done. Best={best_acc:.4f} ({time.time()-t0:.0f}s)")
    return best_acc, best_state

@torch.no_grad()
def evaluate_model(model, loader, label):
    model.eval()
    all_preds = []
    all_labels = []
    all_alpha = []

    alpha_by_class = {i: [] for i in range(NUM_CLASSES)}

    for X_b, phys_b, y_b in loader:
        X_b = X_b.to(device)
        phys_b = phys_b.to(device)

        logits, alpha, _, _, _ = model(X_b, phys_b)
        preds = logits.argmax(dim=1).cpu().numpy()

        all_preds.extend(preds.tolist())
        all_labels.extend(y_b.numpy().tolist())
        all_alpha.extend(alpha.mean(axis=1).cpu().numpy().tolist())

        for a, y in zip(alpha.mean(axis=1).cpu().numpy(), y_b.numpy()):
            alpha_by_class[int(y)].append(float(a))

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_alpha = np.array(all_alpha)

    print(f"\n{'='*70}")
    print(label)
    print(f"{'='*70}")
    print(classification_report(
        all_labels,
        all_preds,
        target_names=ACTIVITY_NAMES,
        digits=4,
        zero_division=0
    ))

    print("Routing statistics:")
    print(f"  mean alpha(time-branch weight): {all_alpha.mean():.4f}")
    print(f"  mean freq weight            : {(1-all_alpha).mean():.4f}")
    for c in range(NUM_CLASSES):
        arr = np.array(alpha_by_class[c]) if len(alpha_by_class[c]) > 0 else np.array([0.0])
        print(f"  {ACTIVITY_NAMES[c]:20s} alpha_mean={arr.mean():.4f}  freq_mean={(1-arr).mean():.4f}")

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    acc = (all_labels == all_preds).mean()
    return acc, macro_f1, all_preds, all_labels, all_alpha

# ============================================================
# 11) Main
# ============================================================
model = PhysicsGuidedPatchDualTransformer(
    num_channels=NUM_CHANNELS,
    patch_size=PATCH_SIZE,
    num_patches=N_PATCHES,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    dropout=DROPOUT,
    num_classes=NUM_CLASSES,
    phys_dim=PHYS_DIM,
    gate_hidden=GATE_HIDDEN,
    num_freq_bands=NUM_FREQ_BANDS,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable params: {n_params:,}")

best_acc, best_state = train_model(
    model, tr_loader, te_loader,
    epochs=EPOCHS,
    lr=LR,
    warmup_ep=WARMUP_EP,
    label="PhysicsGuidedPatchDual"
)

model.load_state_dict(best_state)
acc, macro_f1, preds, labels, alphas = evaluate_model(
    model, te_loader, "Physics-Guided Patch-Level Time/Frequency Dual Transformer"
)

print(f"\nFinal Test Acc     : {acc:.4f}")
print(f"Final Test Macro-F1: {macro_f1:.4f}")
print("Done.")

Device: cuda

Loading UCI-HAR...
Train: (7352, 128, 9) (7352,) subjects: [ 1  3  5  6  7  8 11 14 15 16 17 19 21 22 23 25 26 27 28 29 30]
Test : (2947, 128, 9) (2947,) subjects: [ 2  4  9 10 12 13 18 20 24]
NUM_CLASSES : 6
NUM_CHANNELS: 9
N_PATCHES   : 16

Building patch-level physics descriptors...
Patch physics train: (7352, 16, 9)
Patch physics test : (2947, 16, 9)
PHYS_DIM: 9

Trainable params: 155,695
[PhysicsGuidedPatchDual] ep  1 | loss=1.0725 tr=0.7496 te=0.8656 best=0.8656 alpha_mean=0.515
[PhysicsGuidedPatchDual] ep 10 | loss=0.4541 tr=0.9880 te=0.9413 best=0.9630 alpha_mean=0.516
[PhysicsGuidedPatchDual] ep 20 | loss=0.4350 tr=0.9966 te=0.9623 best=0.9650 alpha_mean=0.515
[PhysicsGuidedPatchDual] ep 30 | loss=0.4266 tr=0.9997 te=0.9664 best=0.9701 alpha_mean=0.504
[PhysicsGuidedPatchDual] ep 40 | loss=0.4261 tr=0.9999 te=0.9681 best=0.9701 alpha_mean=0.498
[PhysicsGuidedPatchDual] ep 50 | loss=0.4261 tr=0.9999 te=0.9681 best=0.9701 alpha_mean=0.495
[PhysicsGuidedPatchDual] D

# IV-E. Robustness to Sensor Perturbations

4가지 강건성 테스트:  
1. **Temporal Scaling** — 시간축 리샘플링 (샘플링 레이트 변동 시뮬레이션)  
2. **Additive Gaussian Noise** — 가우시안 노이즈 추가  
3. **Additive Bias Drift** — 시간에 따라 선형 증가하는 바이어스  
4. **Multiplicative Scale Drift** — 센서 감도 변화 (곱셈 스케일 팩터)  

학습된 모델(best_state)을 그대로 사용하며, **테스트 데이터에만** perturbation을 가합니다.

In [ ]:
# ============================================================
# IV-E. Robustness to Sensor Perturbations
# ============================================================
# Prerequisites: model (loaded with best_state), X_te, y_te,
#                phys_mu, phys_std, PATCH_SIZE, device
# ============================================================

import numpy as np
import torch
import copy
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from sklearn.metrics import f1_score, classification_report

# ── Helper: evaluate perturbed test set ──
def evaluate_perturbed(model, X_perturbed, y_te, phys_mu, phys_std,
                       patch_size=PATCH_SIZE, batch_size=BATCH_SIZE):
    """
    Perturbed X_te -> re-compute physics descriptors -> evaluate.
    Returns (accuracy, macro_f1).
    """
    # Re-compute physics descriptors from perturbed signals
    phys_pert = np.stack(
        [extract_patch_physics_np(x, patch_size) for x in X_perturbed]
    ).astype(np.float32)
    phys_pert = (phys_pert - phys_mu[None, :, :]) / phys_std[None, :, :]

    ds = UCIHARDataset(X_perturbed, y_te, phys_pert)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=2, pin_memory=True)

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_b, phys_b, y_b in loader:
            X_b = X_b.to(device)
            phys_b = phys_b.to(device)
            logits, _, _, _, _ = model(X_b, phys_b)
            all_preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
            all_labels.extend(y_b.numpy().tolist())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    acc = (all_preds == all_labels).mean()
    mf1 = f1_score(all_labels, all_preds, average='macro')
    return acc, mf1


# ============================================================
# 1) Temporal Scaling
#    Simulate sampling-rate variation by resampling the time axis.
#    scale > 1 : signal stretched (slower sampling)
#    scale < 1 : signal compressed (faster sampling)
# ============================================================
def perturb_temporal_scaling(X, scale_factor):
    """
    X: (N, T, C) numpy array
    scale_factor: float, e.g. 0.8 ~ 1.2
    Returns: (N, T, C) — resampled back to original length T
    """
    N, T, C = X.shape
    new_len = int(T * scale_factor)
    if new_len < 2:
        new_len = 2
    X_out = np.zeros_like(X)
    old_t = np.linspace(0, 1, T)
    new_t = np.linspace(0, 1, new_len)
    target_t = np.linspace(0, 1, T)

    for i in range(N):
        for c in range(C):
            # Resample: original -> scaled length -> back to T
            f_interp = interp1d(old_t, X[i, :, c], kind='linear',
                                fill_value='extrapolate')
            scaled_signal = f_interp(new_t)
            f_back = interp1d(np.linspace(0, 1, new_len), scaled_signal,
                              kind='linear', fill_value='extrapolate')
            X_out[i, :, c] = f_back(target_t)
    return X_out


# ============================================================
# 2) Additive Gaussian Noise
#    x' = x + N(0, sigma^2)
# ============================================================
def perturb_gaussian_noise(X, sigma):
    noise = np.random.randn(*X.shape).astype(np.float32) * sigma
    return X + noise


# ============================================================
# 3) Additive Bias Drift
#    Per-channel linear drift: x'(t,c) = x(t,c) + drift_c * (t/T)
#    drift_c ~ U(-magnitude, +magnitude)
# ============================================================
def perturb_bias_drift(X, magnitude):
    N, T, C = X.shape
    X_out = X.copy()
    ramp = np.linspace(0, 1, T).reshape(1, T, 1).astype(np.float32)
    # Per-sample, per-channel random drift direction
    drift = np.random.uniform(-magnitude, magnitude,
                              size=(N, 1, C)).astype(np.float32)
    X_out = X_out + drift * ramp
    return X_out


# ============================================================
# 4) Multiplicative Scale Drift
#    Per-channel scale: x'(t,c) = x(t,c) * s_c
#    s_c ~ U(1 - magnitude, 1 + magnitude)
# ============================================================
def perturb_scale_drift(X, magnitude):
    N, T, C = X.shape
    scale = np.random.uniform(1.0 - magnitude, 1.0 + magnitude,
                              size=(N, 1, C)).astype(np.float32)
    return X * scale


# ============================================================
# Run all 4 robustness tests
# ============================================================
print('='*70)
print('ROBUSTNESS TO SENSOR PERTURBATIONS')
print('='*70)

# Ensure model is loaded with best weights
model.load_state_dict(best_state)
model.eval()

# Baseline (clean test set)
baseline_acc, baseline_f1 = evaluate_perturbed(
    model, X_te, y_te, phys_mu, phys_std
)
print(f'\nBaseline (clean):  Acc={baseline_acc:.4f}  Macro-F1={baseline_f1:.4f}')
print('-'*70)

# ── Test configurations ──
results = {}

# 1) Temporal Scaling
temporal_scales = [0.8, 0.9, 1.1, 1.2]
print('\n[1] Temporal Scaling')
ts_results = []
for sf in temporal_scales:
    X_pert = perturb_temporal_scaling(X_te, sf)
    acc, mf1 = evaluate_perturbed(model, X_pert, y_te, phys_mu, phys_std)
    ts_results.append((sf, acc, mf1))
    print(f'  scale={sf:.1f}  Acc={acc:.4f}  F1={mf1:.4f}  '
          f'(Δacc={acc - baseline_acc:+.4f})')
results['Temporal Scaling'] = ts_results

# 2) Additive Gaussian Noise
noise_sigmas = [0.1, 0.2, 0.5, 1.0]
print('\n[2] Additive Gaussian Noise')
gn_results = []
for sigma in noise_sigmas:
    np.random.seed(SEED)  # reproducible noise
    X_pert = perturb_gaussian_noise(X_te, sigma)
    acc, mf1 = evaluate_perturbed(model, X_pert, y_te, phys_mu, phys_std)
    gn_results.append((sigma, acc, mf1))
    print(f'  sigma={sigma:.1f}  Acc={acc:.4f}  F1={mf1:.4f}  '
          f'(Δacc={acc - baseline_acc:+.4f})')
results['Gaussian Noise'] = gn_results

# 3) Additive Bias Drift
bias_magnitudes = [0.1, 0.2, 0.5, 1.0]
print('\n[3] Additive Bias Drift')
bd_results = []
for mag in bias_magnitudes:
    np.random.seed(SEED)
    X_pert = perturb_bias_drift(X_te, mag)
    acc, mf1 = evaluate_perturbed(model, X_pert, y_te, phys_mu, phys_std)
    bd_results.append((mag, acc, mf1))
    print(f'  magnitude={mag:.1f}  Acc={acc:.4f}  F1={mf1:.4f}  '
          f'(Δacc={acc - baseline_acc:+.4f})')
results['Bias Drift'] = bd_results

# 4) Multiplicative Scale Drift
scale_magnitudes = [0.1, 0.2, 0.3, 0.5]
print('\n[4] Multiplicative Scale Drift')
sd_results = []
for mag in scale_magnitudes:
    np.random.seed(SEED)
    X_pert = perturb_scale_drift(X_te, mag)
    acc, mf1 = evaluate_perturbed(model, X_pert, y_te, phys_mu, phys_std)
    sd_results.append((mag, acc, mf1))
    print(f'  magnitude={mag:.1f}  Acc={acc:.4f}  F1={mf1:.4f}  '
          f'(Δacc={acc - baseline_acc:+.4f})')
results['Scale Drift'] = sd_results

print('\n' + '='*70)
print('DONE — All robustness tests completed.')
print('='*70)


In [ ]:
# ============================================================
# Robustness Results — Summary Table & Visualization
# ============================================================
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 11

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Robustness to Sensor Perturbations', fontsize=15, fontweight='bold')

configs = [
    ('Temporal Scaling',       'Scale Factor', results['Temporal Scaling']),
    ('Gaussian Noise',         'σ (noise std)', results['Gaussian Noise']),
    ('Bias Drift',             'Drift Magnitude', results['Bias Drift']),
    ('Scale Drift',            'Scale Magnitude', results['Scale Drift']),
]

for ax, (title, xlabel, data) in zip(axes.flatten(), configs):
    levels = [d[0] for d in data]
    accs   = [d[1] for d in data]
    f1s    = [d[2] for d in data]

    x_pos = np.arange(len(levels))
    width = 0.35

    bars1 = ax.bar(x_pos - width/2, accs, width, label='Accuracy',
                   color='#4C72B0', alpha=0.85)
    bars2 = ax.bar(x_pos + width/2, f1s, width, label='Macro-F1',
                   color='#DD8452', alpha=0.85)

    # Baseline line
    ax.axhline(y=baseline_acc, color='#4C72B0', linestyle='--',
               alpha=0.5, linewidth=1)
    ax.axhline(y=baseline_f1, color='#DD8452', linestyle='--',
               alpha=0.5, linewidth=1)

    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Score')
    ax.set_xticks(x_pos)
    ax.set_xticklabels([str(l) for l in levels])
    ax.set_ylim(0, 1.05)
    ax.legend(loc='lower left', fontsize=9)

    # Value labels on bars
    for bar in bars1:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.01,
                f'{h:.3f}', ha='center', va='bottom', fontsize=8)
    for bar in bars2:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.01,
                f'{h:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('robustness_results.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Print summary table ──
print('\n' + '='*70)
print(f'{"Perturbation":<25} {"Level":<10} {"Acc":>8} {"F1":>8} {"ΔAcc":>8}')
print('-'*70)
print(f'{"Baseline (clean)":<25} {"-":<10} {baseline_acc:>8.4f} {baseline_f1:>8.4f} {"-":>8}')
print('-'*70)
for name, data_list in [('Temporal Scaling', results['Temporal Scaling']),
                         ('Gaussian Noise', results['Gaussian Noise']),
                         ('Bias Drift', results['Bias Drift']),
                         ('Scale Drift', results['Scale Drift'])]:
    for lvl, acc, f1 in data_list:
        print(f'{name:<25} {lvl:<10.2f} {acc:>8.4f} {f1:>8.4f} {acc-baseline_acc:>+8.4f}')
    print('-'*70)
print('='*70)
